# Implement Gradient Checkpointing

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google, NVIDIA, Tesla

---

### Problem Statement

Gradient checkpointing (also called activation checkpointing or rematerialization) is a critical memory optimization technique for training deep neural networks. During the forward pass, instead of storing all intermediate activations needed for backpropagation, we discard them and **recompute** them during the backward pass.

This trades compute for memory: training becomes ~33% slower but uses O(sqrt(N)) memory instead of O(N) for a network with N layers. This technique is essential for training large language models and other deep architectures that would otherwise not fit in GPU memory.

Your task is to implement gradient checkpointing using a custom `torch.autograd.Function` that saves only inputs during forward and recomputes activations during backward.

---

### Requirements

1. **CheckpointFunction** — A custom `torch.autograd.Function` whose forward saves only inputs (not intermediate activations), and whose backward recomputes the forward pass to recover gradients.
2. **checkpoint** — A wrapper function that applies gradient checkpointing to any callable.
3. **Deep Network** — Build a 20+ layer network, compare memory usage with and without checkpointing.
4. **Correctness** — Gradients with checkpointing must be identical to gradients without.

---

### Constraints

- ✅ Must work with arbitrary functions (not just nn.Module).
- ✅ Must correctly handle the autograd graph (gradients flow through the checkpoint).
- ✅ Gradients must match exactly (bitwise identical to non-checkpointed version).
- ❌ Do **not** use `torch.utils.checkpoint` in your implementation.

---

<details>
  <summary>💡 Hint</summary>

  - In `CheckpointFunction.forward`, use `ctx.save_for_backward(*inputs)` to save only the inputs. Detach the inputs when running the function to prevent PyTorch from building a graph for intermediates.
  - In `CheckpointFunction.backward`, re-enable gradients with `torch.enable_grad()`, rerun the function on the saved inputs, then call `torch.autograd.grad()` to compute gradients of the recomputed outputs w.r.t. the inputs.
  - Save the function itself on the context: `ctx.fn = fn`.
  - Memory estimation: count the number of tensors stored in the autograd graph, or use `torch.cuda.memory_allocated()` on GPU.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Deep network for testing
class DeepBlock(nn.Module):
    """A single residual block: Linear -> ReLU -> Linear + residual."""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, x):
        residual = x
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x + residual


class DeepNetwork(nn.Module):
    """A deep network with many residual blocks."""
    def __init__(self, dim=256, num_blocks=24, use_checkpoint=False):
        super().__init__()
        self.blocks = nn.ModuleList([DeepBlock(dim) for _ in range(num_blocks)])
        self.head = nn.Linear(dim, 1)
        self.use_checkpoint = use_checkpoint

    def forward(self, x):
        for block in self.blocks:
            if self.use_checkpoint:
                x = checkpoint(block, x)
            else:
                x = block(x)
        return self.head(x)


torch.manual_seed(42)
print("Deep network blocks: 24")
print(f"Parameters per block: {sum(p.numel() for p in DeepBlock(256).parameters()):,}")

In [ ]:
class CheckpointFunction(torch.autograd.Function):
    """
    Custom autograd function that implements gradient checkpointing.
    
    Forward: Run the function normally but only save inputs, not intermediates.
    Backward: Recompute the forward pass to get intermediates, then compute gradients.
    """

    @staticmethod
    def forward(ctx, fn, *args):
        """
        Run fn(*args) but only save the inputs for backward.
        
        Args:
            ctx: Context object for saving info between forward and backward.
            fn: The function to checkpoint.
            *args: Arguments to fn (must be tensors).
        
        Returns:
            Output of fn(*args), but without saving intermediate activations.
        """
        # TODO: Save fn on ctx so backward can access it
        # TODO: Save the input tensors using ctx.save_for_backward
        # TODO: Run fn with no_grad to avoid saving intermediates
        # TODO: Return the output
        ...

    @staticmethod
    def backward(ctx, *grad_outputs):
        """
        Recompute forward pass and compute gradients.
        
        Returns:
            None (for fn) + gradients for each input tensor.
        """
        # TODO: Retrieve saved inputs and fn from ctx
        # TODO: Enable gradients and recompute forward pass on the saved inputs
        #       (need to set requires_grad on inputs and use torch.enable_grad)
        # TODO: Use torch.autograd.grad to compute gradients of outputs w.r.t. inputs
        # TODO: Return (None, *grads) — None for the fn argument
        ...


def checkpoint(fn, *args):
    """
    Apply gradient checkpointing to fn(*args).
    
    Args:
        fn: Callable (e.g., a nn.Module or function).
        *args: Input tensors.
    
    Returns:
        Output of fn(*args) with gradient checkpointing applied.
    """
    # TODO: Call CheckpointFunction.apply with fn and args
    ...


def estimate_memory(model, x):
    """
    Estimate activation memory by counting tensors stored for backward.
    Returns a rough count of intermediate activation elements.
    """
    activation_sizes = []

    def hook_fn(module, input, output):
        if isinstance(output, torch.Tensor):
            activation_sizes.append(output.numel())

    hooks = []
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.ReLU)):
            hooks.append(module.register_forward_hook(hook_fn))

    model(x)

    for h in hooks:
        h.remove()

    return sum(activation_sizes)

In [ ]:
# === Validation ===
torch.manual_seed(42)

# 1. Gradient correctness test
print("=== Gradient Correctness Test ===")
dim = 256
x = torch.randn(16, dim, requires_grad=True)

# Without checkpointing
model_no_ckpt = DeepNetwork(dim=dim, num_blocks=24, use_checkpoint=False)
torch.manual_seed(42)
model_ckpt = DeepNetwork(dim=dim, num_blocks=24, use_checkpoint=True)
# Copy weights
model_ckpt.load_state_dict(model_no_ckpt.state_dict())

# Forward + backward without checkpointing
x1 = x.detach().clone().requires_grad_(True)
out1 = model_no_ckpt(x1)
loss1 = out1.sum()
loss1.backward()

# Forward + backward with checkpointing
x2 = x.detach().clone().requires_grad_(True)
out2 = model_ckpt(x2)
loss2 = out2.sum()
loss2.backward()

# Compare gradients
print(f"Output match: {torch.allclose(out1, out2, atol=1e-6)}")
print(f"Input grad match: {torch.allclose(x1.grad, x2.grad, atol=1e-5)}")

grad_match = True
for (n1, p1), (n2, p2) in zip(model_no_ckpt.named_parameters(), model_ckpt.named_parameters()):
    if p1.grad is not None and p2.grad is not None:
        if not torch.allclose(p1.grad, p2.grad, atol=1e-5):
            print(f"  Gradient mismatch at {n1}: max diff = {(p1.grad - p2.grad).abs().max():.2e}")
            grad_match = False

assert grad_match, "Gradients should match between checkpointed and non-checkpointed!"
print("All parameter gradients match!")
print("PASSED\n")

# 2. Memory estimation test
print("=== Memory Estimation Test ===")
x_test = torch.randn(16, dim)
mem_no_ckpt = estimate_memory(model_no_ckpt, x_test)
mem_ckpt = estimate_memory(model_ckpt, x_test)
print(f"Activation elements without checkpointing: {mem_no_ckpt:,}")
print(f"Activation elements with checkpointing:    {mem_ckpt:,}")
print(f"Note: The forward hooks count ALL activations produced during forward.")
print(f"With checkpointing, intermediates are recomputed during backward, not stored.")
print("PASSED\n")

# 3. Training works correctly
print("=== Training Test ===")
model_train = DeepNetwork(dim=64, num_blocks=24, use_checkpoint=True)
optimizer = torch.optim.Adam(model_train.parameters(), lr=1e-3)
X = torch.randn(32, 64)
y = torch.randn(32, 1)

losses = []
for step in range(30):
    optimizer.zero_grad()
    pred = model_train(X)
    loss = F.mse_loss(pred, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss should decrease with checkpointed training!"
print("PASSED\n")

print("All tests passed!")